In [3]:
import pandas as pd

f1 = pd.read_csv("1200 questions.csv")
f2 = pd.read_csv("2nd part 1200.csv")
f3 = pd.read_csv("3rd part.csv")

for df in [f1, f2, f3]:
    df.columns = df.columns.str.strip()

    if "Questions" in df.columns:
        df.rename(columns={"Questions": "Question"}, inplace=True)

df = pd.concat([f1, f2, f3], ignore_index=True)
df.to_csv("dataset.csv", index=False)

In [4]:
df = pd.read_csv("dataset.csv")
df.head()

,Question,Answer
0,Which learning concept describes Leaky negativ...,LeakyReLU
1,What do we call Classification accuracy metric?,Accuracy
2,What is the name for Transformer input represe...,Embedding
3,Which learning concept describes Activation ce...,Tanh
4,Which component handles Bidirectional recurren...,BiRNN


In [5]:
df.Question[0]

'Which learning concept describes Leaky negative ReLU?'

In [6]:
df['Question'] = df['Question'].str.lower().str.replace(r'[\?!."\']', '', regex=True).str.replace(r'\s+', ' ',regex=True).str.strip()
df['Answer'] = df['Answer'].str.lower().str.replace(r'[\?!."\']', '', regex=True).str.replace(r'\s+', ' ',regex=True).str.strip()

df

,Question,Answer
0,which learning concept describes leaky negativ...,leakyrelu
1,what do we call classification accuracy metric,accuracy
2,what is the name for transformer input represe...,embedding
3,which learning concept describes activation ce...,tanh
4,which component handles bidirectional recurren...,birnn
...,...,...
3295,which organization or person is associated wit...,chen
3296,which organization or person is associated wit...,microsoft
3297,which organization or person is associated wit...,yandex
3298,which organization or person is associated wit...,openai


In [7]:
df['Question'][0]

'which learning concept describes leaky negative relu'

In [8]:
import re
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [11]:
def tokenize(text):
    return text.split()

print(tokenize(df['Question'][0]))

['which', 'learning', 'concept', 'describes', 'leaky', 'negative', 'relu']


In [12]:
vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for _, row in df.iterrows():
    question_tokens = tokenize(row['Question'])
    answer_tokens = tokenize(row['Answer'])

    tokens = question_tokens + answer_tokens
    for token in tokens:
        if token not in vocab:
            vocab[token] = len(vocab)

index_to_word = {index: word for word, index in vocab.items()}
print(len(vocab))

3157


In [13]:
print(list(vocab.items())[:10])

[('<PAD>', 0), ('<UNK>', 1), ('which', 2), ('learning', 3), ('concept', 4), ('describes', 5), ('leaky', 6), ('negative', 7), ('relu', 8), ('leakyrelu', 9)]


In [14]:
def text_to_indices(text, vocab):
    tokens = tokenize(text)
    indices = [vocab.get(token, vocab["<UNK>"]) for token in tokens]
    return indices

In [15]:
text_to_indices(df['Question'][111], vocab)

[10, 17, 65, 296, 131]

In [16]:
class QADataset(Dataset):
    def __init__(self, df, vocab):
        self.df = df
        self.vocab = vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        question = self.df.iloc[index]['Question']
        answer = self.df.iloc[index]['Answer']

        question_indices = text_to_indices(question, self.vocab)
        answer_indices = text_to_indices(answer, self.vocab)

        return torch.tensor(question_indices), torch.tensor(answer_indices)

In [17]:
dataset = QADataset(df, vocab)

In [ ]:
dataloader = DataLoader(dataset, batch_size=, shuffle=True)

sample_question, sample_answer = dataset[2]
print("Sample Question Indices:", sample_question)
print("Sample Answer Indices:", sample_answer)

Sample Question Indices: tensor([10, 17, 18, 19, 20, 21, 22, 23])
Sample Answer Indices: tensor([24])


In [19]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(SimpleRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)
        output = self.fc(hidden.squeeze(0))
        return output

model = SimpleRNN(vocab_size=len(vocab), embedding_dim=60, hidden_dim=100, output_dim=len(vocab))
print(model)

SimpleRNN(
  (embedding): Embedding(3157, 60)
  (rnn): RNN(60, 100, batch_first=True)
  (fc): Linear(in_features=100, out_features=3157, bias=True)
)


In [20]:
dummy_input = torch.tensor([text_to_indices(df['Question'][0], vocab)])
dummy_output = model(dummy_input)
print("Dummy Output Shape:", dummy_output.shape)
print("Dummy Output:", dummy_output)

Dummy Output Shape: torch.Size([1, 3157])
Dummy Output: tensor([[ 0.6225,  0.4065,  0.0225,  ...,  0.5012, -0.5427, -0.3455]],
       grad_fn=<AddmmBackward0>)


In [22]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):

    total_loss = 0.0
    for question_indices, answer_indices in dataloader:
        optimizer.zero_grad()
        outputs = model(question_indices)
        answer_indices = answer_indices.squeeze(0)
        loss = criterion(outputs, answer_indices)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader)}")

print("Hurreh! Training complete.")

Epoch 1, Loss: 5.748394463437073
Epoch 2, Loss: 3.1692449294273373
Epoch 3, Loss: 2.051285949942769
Epoch 4, Loss: 1.3042403663536817
Epoch 5, Loss: 0.7793434194465888
Epoch 6, Loss: 0.46606640416347406
Epoch 7, Loss: 0.29547829321959096
Epoch 8, Loss: 0.20027254142866172
Epoch 9, Loss: 0.1710389926177992
Epoch 10, Loss: 0.1382254975671502
Hurreh! Training complete.


In [23]:
def predict(question_text, model, vocab, idx_to_word):
  model.eval()
  with torch.no_grad():
    indices = text_to_indices(question_text, vocab)
    if not indices:
      return "<UNK>"

    x = torch.tensor(indices, dtype=torch.long).unsqueeze(0)

    prediction = model(x)
    pred_idx = torch.argmax(prediction, dim=1).item()

    return idx_to_word.get(pred_idx, "<UNK>")



tests = [
    (
        "Which NLP task condenses lengthy text into a concise overview?",
        "Summarization",
    ),
    (
        "In binary classification, what error label denotes a missed positive"
        " detection?",
        "FN",
    ),
    (
        "Which text preparation step segments sentences into individual units?",
        "Tokenization",
    ),
    (
        "Which pooling layer extracts the highest activation value across a"
        " feature window?",
        "MaxPool",
    ),
    (
        "Which structural property enables an RNN to maintain history across"
        " time steps?",
        "Recurrence",
    ),
    (
        "What PyTorch dictionary object stores learned parameters and"
        " persistent buffers?",
        "StateDict",
    ),
    (
        "Name the distributed tool used to track source code history and"
        " commits.",
        "Git",
    ),
    (
        "Which attention component vector interacts with queries to produce"
        " alignment weights?",
        "Key",
    ),
]


for question, expected in tests:
  prediction = predict(question, model, vocab, index_to_word)
  print(f"Q: {question}")
  print(f"   Expected  : {expected}")
  print(f"   Predicted : {prediction}")
  print("-" * 60)

Q: Which NLP task condenses lengthy text into a concise overview?
   Expected  : Summarization
   Predicted : no
------------------------------------------------------------
Q: In binary classification, what error label denotes a missed positive detection?
   Expected  : FN
   Predicted : no
------------------------------------------------------------
Q: Which text preparation step segments sentences into individual units?
   Expected  : Tokenization
   Predicted : yes
------------------------------------------------------------
Q: Which pooling layer extracts the highest activation value across a feature window?
   Expected  : MaxPool
   Predicted : yes
------------------------------------------------------------
Q: Which structural property enables an RNN to maintain history across time steps?
   Expected  : Recurrence
   Predicted : no
------------------------------------------------------------
Q: What PyTorch dictionary object stores learned parameters and persistent buffers?
   E